# LeetCode 716 — Max Stack bằng Python

**Link đề:** https://leetcode.com/problems/max-stack/
**Độ khó:** Easy theo danh mục repo (LeetCode hiện ghi **Hard**) · **Chủ đề:** Stack, Design, Linked List, Doubly-Linked List, Ordered Set, Heap

> **Bài Premium: đề bài trong notebook được viết theo hiểu biết về bài, nên đối chiếu lại nếu có tài khoản Premium.**
>
> Ghi chú độ khó: notebook được xếp vào `DSA/Easy` theo danh mục của repo, dù LeetCode hiện ghi bài này là Hard (vì follow-up yêu cầu `popMax` nhanh).

```python
class MaxStack:

    def __init__(self):
        ...

    def push(self, x: int) -> None:
        ...

    def pop(self) -> int:
        ...

    def top(self) -> int:
        ...

    def peekMax(self) -> int:
        ...

    def popMax(self) -> int:
        ...
```

## Mục tiêu học
1. Hiểu **stack** (LIFO) và yêu cầu thêm: biết/lấy ra phần tử **lớn nhất** bất kỳ lúc nào.
2. Đi từ cách quét tuyến tính → **hai stack (stack + stack max tiền tố)** → **heap + xóa lười (lazy deletion)** đạt `O(log n)`.
3. Hiểu cách phá hòa khi có nhiều max: `popMax` phải lấy phần tử max **gần đỉnh nhất** → dùng **id tăng dần** (timestamp).
4. Trace trạng thái stack/heap/tập đã xóa sau mỗi lệnh; chứng minh invariant; phân tích amortized.
5. Biến thể: Min Stack (155), Max Queue, frequency stack.

## 1. Đề bài (theo hiểu biết về bài)

Thiết kế cấu trúc dữ liệu **stack** hỗ trợ thêm thao tác lấy phần tử lớn nhất:

- `MaxStack()`: khởi tạo stack rỗng.
- `push(x)`: đẩy `x` lên đỉnh stack.
- `pop()`: xóa phần tử ở **đỉnh** và trả về nó.
- `top()`: trả về phần tử ở đỉnh (không xóa).
- `peekMax()`: trả về phần tử **lớn nhất** trong stack (không xóa).
- `popMax()`: tìm phần tử lớn nhất, **xóa** nó khỏi stack và trả về. Nếu có **nhiều** phần tử cùng bằng max, chỉ xóa phần tử **gần đỉnh nhất** (top-most).

**Follow-up:** `top` trong `O(1)`, các thao tác còn lại `O(log n)`.

**Ví dụ:**
```text
Input:  ["MaxStack","push","push","push","top","popMax","top","peekMax","pop","top"]
        [[],[5],[1],[5],[],[],[],[],[],[]]
Output: [null,null,null,null,5,5,1,5,1,5]
```

| Lệnh | Stack (đáy → đỉnh) | Trả về | Giải thích |
|---|---|:---:|---|
| push(5) | [5] | | |
| push(1) | [5, 1] | | |
| push(5) | [5, 1, 5] | | |
| top() | [5, 1, 5] | 5 | đỉnh là 5 |
| popMax() | [5, 1] | 5 | có hai số 5, xóa số 5 **ở đỉnh** |
| top() | [5, 1] | 1 | |
| peekMax() | [5, 1] | 5 | |
| pop() | [5] | 1 | |
| top() | [5] | 5 | |

**Constraints (theo hiểu biết):** `-10^7 <= x <= 10^7`; tối đa `10^5` lời gọi; khi gọi `pop`, `top`, `peekMax`, `popMax` thì stack luôn có ít nhất một phần tử.

In [1]:
OPS = ["MaxStack", "push", "push", "push", "top", "popMax", "top", "peekMax", "pop", "top"]
ARGS = [[], [5], [1], [5], [], [], [], [], [], []]
EXP = [None, None, None, None, 5, 5, 1, 5, 1, 5]

def run_ops(cls, ops, args):
    out, obj = [], None
    for op, a in zip(ops, args):
        if op == ops[0]:
            obj = cls(*a)
            out.append(None)
        else:
            out.append(getattr(obj, op)(*a))
    return out

print("Kịch bản mẫu có", len(OPS), "lệnh")

Kịch bản mẫu có 10 lệnh


`run_ops` chạy kịch bản kiểu LeetCode: lệnh đầu tạo đối tượng, các lệnh sau gọi phương thức theo tên (`getattr`). `push` không trả gì nên Python trả `None` — khớp với `null`.

## 2. Nền tảng: stack bằng list Python

Stack = "chồng đĩa": vào sau ra trước (**LIFO** — Last In First Out). Trong Python dùng `list`:
- `append(x)`: đẩy lên đỉnh — `O(1)`.
- `pop()`: lấy ra từ đỉnh — `O(1)`.
- `st[-1]`: xem đỉnh — `O(1)`.

In [2]:
st = []
st.append(5); st.append(1); st.append(5)
print("stack:", st, "| đỉnh:", st[-1])
print("pop ->", st.pop(), "| còn:", st)

stack: [5, 1, 5] | đỉnh: 5
pop -> 5 | còn: [5, 1]


### 2.1. Xóa ở giữa list tốn `O(n)`

`del st[i]` hoặc `st.pop(i)` phải dịch mọi phần tử sau `i` sang trái. Đây là lý do `popMax` "ngây thơ" chậm.

In [3]:
st = [5, 1, 9, 2, 7]
i = st.index(9)
del st[i]
print(st)

[5, 1, 2, 7]


### 2.2. Tìm chỉ số **cuối cùng** của giá trị lớn nhất

`popMax` cần max **gần đỉnh nhất** = lần xuất hiện **cuối** trong list. Cách viết: duyệt ngược từ cuối.

In [4]:
st = [5, 1, 5, 2]
m = max(st)
last = len(st) - 1 - st[::-1].index(m)
print("max =", m, "| chỉ số cuối =", last)
for i in range(len(st) - 1, -1, -1):        # range đi lùi: start, stop (không gồm), step = -1
    if st[i] == m:
        print("duyệt ngược tìm thấy tại", i)
        break

max = 5 | chỉ số cuối = 2
duyệt ngược tìm thấy tại 2


`range(len(st) - 1, -1, -1)` sinh `3, 2, 1, 0`: bắt đầu từ chỉ số cuối, dừng **trước** `-1`, bước `-1`.

### 2.3. Heap (`heapq`) và mẹo max-heap bằng số âm

`heapq` là **min-heap**: `heap[0]` luôn nhỏ nhất, `heappush`/`heappop` tốn `O(log n)`. Muốn max-heap: đẩy `-x`.

Có thể đẩy **tuple**: heap so sánh phần tử đầu, hòa thì so phần tử kế tiếp. Ta sẽ đẩy `(-x, -id)`: max `x` trước, hòa thì `id` **lớn** (vào sau, gần đỉnh hơn) trước.

In [5]:
import heapq
h = []
for idx, x in enumerate([5, 1, 5]):
    heapq.heappush(h, (-x, -idx))
print("heap:", h)
neg_x, neg_id = h[0]
print("max =", -neg_x, "| id =", -neg_id, "(số 5 vào sau cùng)")

heap: [(-5, -2), (-1, -1), (-5, 0)]
max = 5 | id = 2 (số 5 vào sau cùng)


### 2.4. `set` để đánh dấu "đã xóa"

`removed = set()`; `removed.add(id)`; `id in removed` — trung bình `O(1)`. Dùng cho kỹ thuật **xóa lười**.

## 3. Cách 1 — Một list, quét tuyến tính cho max

Ý tưởng tự nhiên nhất:
- `push/pop/top`: thao tác stack bình thường `O(1)`.
- `peekMax`: `max(stack)` — `O(n)`.
- `popMax`: tìm chỉ số **cuối** của max rồi `del` — `O(n)`.

In [6]:
class MaxStackList:
    def __init__(self):
        self.st = []
    def push(self, x):
        self.st.append(x)
    def pop(self):
        return self.st.pop()
    def top(self):
        return self.st[-1]
    def peekMax(self):
        return max(self.st)
    def popMax(self):
        m = max(self.st)
        for i in range(len(self.st) - 1, -1, -1):
            if self.st[i] == m:
                del self.st[i]
                return m

print(run_ops(MaxStackList, OPS, ARGS))
print(run_ops(MaxStackList, OPS, ARGS) == EXP)

[None, None, None, None, 5, 5, 1, 5, 1, 5]
True


**Nút thắt:** `peekMax` và `popMax` đều `O(n)`. Với `10^5` lời gọi trên stack lớn → cỡ `10^10` bước trường hợp xấu.

## 4. Cách 2 — Hai stack: `stack` + `max_stack` (max tiền tố)

Ý tưởng như bài **Min Stack (155)**: song song với stack chính, giữ `max_stack[i] = max(stack[0..i])`.
- `push(x)`: `max_stack.append(max(x, max_stack[-1]))` — `O(1)`.
- `pop()`: pop cả hai — `O(1)`.
- `peekMax()`: `max_stack[-1]` — `O(1)`.
- `popMax()`: pop tạm ra một **buffer** cho tới khi gặp max, bỏ max đó, rồi push lại buffer — `O(n)` xấu nhất.

In [7]:
class MaxStackTwo:
    def __init__(self):
        self.st = []
        self.mx = []
    def push(self, x):
        self.st.append(x)
        self.mx.append(x if not self.mx else max(x, self.mx[-1]))
    def pop(self):
        self.mx.pop()
        return self.st.pop()
    def top(self):
        return self.st[-1]
    def peekMax(self):
        return self.mx[-1]
    def popMax(self):
        m = self.mx[-1]
        buffer = []
        while self.top() != m:          # pop cho tới khi đỉnh là max
            buffer.append(self.pop())
        self.pop()                      # bỏ chính phần tử max (gần đỉnh nhất)
        while buffer:
            self.push(buffer.pop())     # push lại, max_stack tự tính lại
        return m

print(run_ops(MaxStackTwo, OPS, ARGS))
print(run_ops(MaxStackTwo, OPS, ARGS) == EXP)

[None, None, None, None, 5, 5, 1, 5, 1, 5]
True


Vì sao `popMax` lấy đúng max **gần đỉnh nhất**? Ta pop từ đỉnh xuống, dừng ở phần tử **đầu tiên** bằng `m` gặp được — chính là phần tử max gần đỉnh nhất.

Đây đã tốt hơn nhiều (4/5 thao tác `O(1)`), nhưng `popMax` vẫn `O(n)`, chưa đạt follow-up.

## 5. Insight — heap + xóa lười (lazy deletion)

Muốn `popMax` nhanh, cần một cấu trúc trả về max nhanh **và** xóa được: **max-heap**. Nhưng mỗi phần tử nằm ở **hai** nơi (stack và heap); xóa ở nơi này thì nơi kia vẫn còn bản "ma".

**Xóa lười:** không xóa ngay ở cấu trúc kia, chỉ **đánh dấu** `id` của phần tử vào tập `removed`. Khi cần đọc đỉnh của stack hoặc heap, nếu đỉnh là phần tử đã bị đánh dấu thì **dọn** (pop bỏ) cho tới khi đỉnh hợp lệ.

Cần `id` duy nhất cho mỗi lần `push` (vì có thể push trùng giá trị), lấy từ bộ đếm tăng dần. `id` lớn hơn = vào sau = gần đỉnh hơn → dùng để phá hòa trong heap: `(-x, -id)`.

In [8]:
stack = [(5, 0), (1, 1), (5, 2)]        # (giá trị, id)
heap = [(-5, -2), (-5, 0), (-1, -1)]
heapq.heapify(heap)
removed = set()

# popMax: lấy từ heap
neg_x, neg_id = heapq.heappop(heap)
removed.add(-neg_id)
print("popMax lấy", -neg_x, "id", -neg_id, "| removed =", removed)
# stack vẫn còn (5, 2) ở đỉnh -> khi top() phải dọn:
while stack[-1][1] in removed:
    print("  dọn đỉnh stack:", stack.pop())
print("top() =", stack[-1][0])

popMax lấy 5 id 2 | removed = {2}
  dọn đỉnh stack: (5, 2)
top() = 1


## 6. Cách 3 — Heap + xóa lười (lời giải chính, `O(log n)` amortized)

Mổ xẻ từng phần:

- **Trạng thái:** `self.stack` (list các `(x, id)`), `self.heap` (các `(-x, -id)`), `self.removed` (set id đã xóa), `self.cnt` (bộ đếm id).
- **`_clean_stack()`**: trong khi đỉnh stack có id thuộc `removed` → pop bỏ. Một id bị đánh dấu chỉ còn "ma" ở **đúng một** cấu trúc, nên dọn xong thì `discard` nó khỏi `removed` để set không phình mãi.
- **`_clean_heap()`**: tương tự cho đỉnh heap.
- `push(x)`: thêm vào cả hai, `cnt += 1`.
- `pop()`: dọn stack → pop đỉnh → đánh dấu id đã xóa (heap còn bản ma).
- `top()`: dọn stack → trả đỉnh.
- `peekMax()`: dọn heap → trả `-heap[0][0]`.
- `popMax()`: dọn heap → heappop → đánh dấu id (stack còn bản ma).

In [9]:
class MaxStackHeap:
    def __init__(self):
        self.stack = []        # (x, id)
        self.heap = []         # (-x, -id)
        self.removed = set()
        self.cnt = 0

    def push(self, x):
        self.stack.append((x, self.cnt))
        heapq.heappush(self.heap, (-x, -self.cnt))
        self.cnt += 1

    def _clean_stack(self):
        while self.stack and self.stack[-1][1] in self.removed:
            self.removed.discard(self.stack.pop()[1])

    def _clean_heap(self):
        while self.heap and -self.heap[0][1] in self.removed:
            self.removed.discard(-heapq.heappop(self.heap)[1])

    def pop(self):
        self._clean_stack()
        x, i = self.stack.pop()
        self.removed.add(i)
        return x

    def top(self):
        self._clean_stack()
        return self.stack[-1][0]

    def peekMax(self):
        self._clean_heap()
        return -self.heap[0][0]

    def popMax(self):
        self._clean_heap()
        neg_x, neg_i = heapq.heappop(self.heap)
        self.removed.add(-neg_i)
        return -neg_x

print(run_ops(MaxStackHeap, OPS, ARGS))
print(run_ops(MaxStackHeap, OPS, ARGS) == EXP)

[None, None, None, None, 5, 5, 1, 5, 1, 5]
True


Giải thích các chỗ dễ vấp:
- `self.stack[-1][1]`: phần tử đỉnh là tuple `(x, id)`, `[1]` là `id`.
- `-self.heap[0][1]`: trong heap lưu `-id`, nên lấy lại `id` bằng dấu trừ.
- `removed.discard(...)` sau khi dọn: id đó đã biến mất ở **cả hai** cấu trúc, không cần nhớ nữa → set luôn nhỏ.
- `self.stack` và `self.heap` có thể chứa phần tử "ma", nhưng ta **luôn dọn trước khi đọc đỉnh**, nên kết quả trả về luôn hợp lệ.

## 7. TRACE từng bước — kịch bản mẫu

| Lệnh | stack (x,id) | heap (đã sắp để dễ đọc) | removed | Trả |
|---|---|---|---|:---:|
| push(5) | (5,0) | (-5,0) | {} | |
| push(1) | (5,0),(1,1) | (-5,0),(-1,-1) | {} | |
| push(5) | (5,0),(1,1),(5,2) | (-5,-2),(-5,0),(-1,-1) | {} | |
| top() | đỉnh (5,2) hợp lệ | | {} | 5 |
| popMax() | không đổi (còn ma (5,2)) | pop (-5,-2) | {2} | 5 |
| top() | dọn (5,2) → đỉnh (1,1) | | {} | 1 |
| peekMax() | | đỉnh (-5,0) hợp lệ | {} | 5 |
| pop() | pop (1,1) | còn ma (-1,-1) | {1} | 1 |
| top() | đỉnh (5,0) | | {1} | 5 |

In [10]:
class MaxStackTrace(MaxStackHeap):
    def _show(self, op, res):
        print(f"{op:10} -> {str(res):5} | stack={self.stack} | heap={sorted(self.heap)} | removed={self.removed}")
    def push(self, x):
        super().push(x); self._show(f"push({x})", "")
    def pop(self):
        r = super().pop(); self._show("pop()", r); return r
    def top(self):
        r = super().top(); self._show("top()", r); return r
    def peekMax(self):
        r = super().peekMax(); self._show("peekMax()", r); return r
    def popMax(self):
        r = super().popMax(); self._show("popMax()", r); return r

_ = run_ops(MaxStackTrace, OPS, ARGS)

push(5)    ->       | stack=[(5, 0)] | heap=[(-5, 0)] | removed=set()
push(1)    ->       | stack=[(5, 0), (1, 1)] | heap=[(-5, 0), (-1, -1)] | removed=set()
push(5)    ->       | stack=[(5, 0), (1, 1), (5, 2)] | heap=[(-5, -2), (-5, 0), (-1, -1)] | removed=set()
top()      -> 5     | stack=[(5, 0), (1, 1), (5, 2)] | heap=[(-5, -2), (-5, 0), (-1, -1)] | removed=set()
popMax()   -> 5     | stack=[(5, 0), (1, 1), (5, 2)] | heap=[(-5, 0), (-1, -1)] | removed={2}
top()      -> 1     | stack=[(5, 0), (1, 1)] | heap=[(-5, 0), (-1, -1)] | removed=set()
peekMax()  -> 5     | stack=[(5, 0), (1, 1)] | heap=[(-5, 0), (-1, -1)] | removed=set()
pop()      -> 1     | stack=[(5, 0)] | heap=[(-5, 0), (-1, -1)] | removed={1}
top()      -> 5     | stack=[(5, 0)] | heap=[(-5, 0), (-1, -1)] | removed={1}


Trace kịch bản thứ hai — nhiều max trùng và xen kẽ `pop`/`popMax`:

In [11]:
ops2 = ["MaxStack", "push", "push", "push", "push", "popMax", "popMax", "pop", "peekMax", "top"]
args2 = [[], [3], [7], [7], [2], [], [], [], [], []]
_ = run_ops(MaxStackTrace, ops2, args2)

push(3)    ->       | stack=[(3, 0)] | heap=[(-3, 0)] | removed=set()
push(7)    ->       | stack=[(3, 0), (7, 1)] | heap=[(-7, -1), (-3, 0)] | removed=set()
push(7)    ->       | stack=[(3, 0), (7, 1), (7, 2)] | heap=[(-7, -2), (-7, -1), (-3, 0)] | removed=set()
push(2)    ->       | stack=[(3, 0), (7, 1), (7, 2), (2, 3)] | heap=[(-7, -2), (-7, -1), (-3, 0), (-2, -3)] | removed=set()
popMax()   -> 7     | stack=[(3, 0), (7, 1), (7, 2), (2, 3)] | heap=[(-7, -1), (-3, 0), (-2, -3)] | removed={2}
popMax()   -> 7     | stack=[(3, 0), (7, 1), (7, 2), (2, 3)] | heap=[(-3, 0), (-2, -3)] | removed={1, 2}
pop()      -> 2     | stack=[(3, 0), (7, 1), (7, 2)] | heap=[(-3, 0), (-2, -3)] | removed={1, 2, 3}
peekMax()  -> 3     | stack=[(3, 0), (7, 1), (7, 2)] | heap=[(-3, 0), (-2, -3)] | removed={1, 2, 3}
top()      -> 3     | stack=[(3, 0)] | heap=[(-3, 0), (-2, -3)] | removed={3}


## 8. Code nộp LeetCode

In [12]:
import heapq

class MaxStack:

    def __init__(self):
        self.stack = []          # (x, id)
        self.heap = []           # (-x, -id)
        self.removed = set()
        self.cnt = 0

    def push(self, x: int) -> None:
        self.stack.append((x, self.cnt))
        heapq.heappush(self.heap, (-x, -self.cnt))
        self.cnt += 1

    def _clean_stack(self) -> None:
        while self.stack and self.stack[-1][1] in self.removed:
            self.removed.discard(self.stack.pop()[1])

    def _clean_heap(self) -> None:
        while self.heap and -self.heap[0][1] in self.removed:
            self.removed.discard(-heapq.heappop(self.heap)[1])

    def pop(self) -> int:
        self._clean_stack()
        x, i = self.stack.pop()
        self.removed.add(i)
        return x

    def top(self) -> int:
        self._clean_stack()
        return self.stack[-1][0]

    def peekMax(self) -> int:
        self._clean_heap()
        return -self.heap[0][0]

    def popMax(self) -> int:
        self._clean_heap()
        x, i = heapq.heappop(self.heap)
        self.removed.add(-i)
        return -x

print(run_ops(MaxStack, OPS, ARGS) == EXP)

True


Về follow-up "`top` trong `O(1)`": với xóa lười, `top` là `O(1)` **amortized** (mỗi phần tử ma chỉ bị dọn một lần). Muốn `O(1)` chặt và `popMax` `O(log n)` chặt, cách kinh điển là **danh sách liên kết đôi + cây cân bằng (TreeMap: giá trị → list node)** như Java; Python chuẩn không có TreeMap (thư viện ngoài `sortedcontainers` có), nên heap + xóa lười là lựa chọn thực tế.

## 9. Cách 4 — Danh sách liên kết đôi + "TreeMap" mô phỏng

Để hiểu cách kinh điển: stack là **doubly linked list** (xóa một node bất kỳ `O(1)` khi có con trỏ tới nó); thêm một map `giá trị → list các node` (node cuối list là node gần đỉnh nhất) và một **danh sách khóa đã sắp xếp** (dùng `bisect`) để biết giá trị lớn nhất. Ở đây chèn khóa mới vào list sắp xếp tốn `O(k)` (k = số giá trị khác nhau) — với TreeMap thật sẽ là `O(log n)`.

In [13]:
import bisect

class DNode:
    def __init__(self, val):
        self.val, self.prev, self.next = val, None, None

class MaxStackDLL:
    def __init__(self):
        self.head, self.tail = DNode(0), DNode(0)      # hai node giả
        self.head.next, self.tail.prev = self.tail, self.head
        self.nodes = {}                                # giá trị -> list node (theo thứ tự push)
        self.keys = []                                 # các giá trị khác nhau, tăng dần

    def _unlink(self, node):
        node.prev.next, node.next.prev = node.next, node.prev

    def push(self, x):
        node = DNode(x)
        node.prev, node.next = self.tail.prev, self.tail
        self.tail.prev.next = node
        self.tail.prev = node
        if x not in self.nodes:
            self.nodes[x] = []
            bisect.insort(self.keys, x)
        self.nodes[x].append(node)

    def _remove_from_map(self, x):
        self.nodes[x].pop()
        if not self.nodes[x]:
            del self.nodes[x]
            self.keys.pop(bisect.bisect_left(self.keys, x))

    def pop(self):
        node = self.tail.prev
        self._unlink(node)
        self._remove_from_map(node.val)       # node đỉnh luôn là node cuối trong list của giá trị đó
        return node.val

    def top(self):
        return self.tail.prev.val

    def peekMax(self):
        return self.keys[-1]

    def popMax(self):
        x = self.keys[-1]
        node = self.nodes[x][-1]              # node max gần đỉnh nhất
        self._unlink(node)
        self._remove_from_map(x)
        return x

print(run_ops(MaxStackDLL, OPS, ARGS) == EXP)

True


Vì sao trong `pop`, node đỉnh luôn là node **cuối** trong `self.nodes[x]`? Các node cùng giá trị được append theo thứ tự push; node đỉnh là node được push muộn nhất **trong số còn lại**, nên nằm cuối list.

## 10. Vì sao đúng? (cách heap + xóa lười)

Gọi tập **sống** `L` = các id đã push mà chưa bị `pop`/`popMax`.

**Invariant:**
1. Mọi id trong `L` có mặt ở **cả** `stack` và `heap`.
2. Mọi phần tử "ma" (id ∉ `L`) còn trong `stack` hoặc `heap` đều có id ∈ `removed`; và id ∈ `removed` ⇔ nó đã bị xóa nhưng còn ma ở đúng **một** cấu trúc.
3. Thứ tự trong `stack` (bỏ qua ma) đúng là thứ tự push của `L`.

Hệ quả:
- Sau `_clean_stack()`, đỉnh stack có id ∈ `L` và là id lớn nhất của `L` nằm trong stack (tức phần tử gần đỉnh nhất) → `top`/`pop` đúng.
- Sau `_clean_heap()`, `heap[0]` là phần tử sống có `(-x, -id)` nhỏ nhất ⇔ `x` lớn nhất, hòa thì `id` lớn nhất (gần đỉnh nhất) → `peekMax`/`popMax` đúng kể cả khi có nhiều max.
- `pop`/`popMax` xóa phần tử khỏi một cấu trúc và đánh dấu id → giữ invariant 2. Khi dọn bản ma ở cấu trúc còn lại, id được `discard` → giữ "⇔".

## 11. Độ phức tạp và so sánh

Gọi `n` = số phần tử đã push, `k` = số giá trị khác nhau.

| Cách | push | pop | top | peekMax | popMax | Bộ nhớ |
|---|---|---|---|---|---|---|
| 1. List quét | `O(1)` | `O(1)` | `O(1)` | `O(n)` | `O(n)` | `O(n)` |
| 2. Hai stack | `O(1)` | `O(1)` | `O(1)` | `O(1)` | `O(n)` | `O(n)` |
| 3. Heap + xóa lười | `O(log n)` | `O(log n)`* | `O(1)`* | `O(log n)`* | `O(log n)`* | `O(n)` |
| 4. DLL + map + bisect | `O(k)`** | `O(k)`** | `O(1)` | `O(1)` | `O(k)`** | `O(n)` |

\* amortized: tổng công việc dọn ma ≤ số phần tử từng push, mỗi lần dọn heap `O(log n)`.
\*\* chỉ khi xuất hiện giá trị mới / giá trị biến mất (insort/pop trên list khóa); với TreeMap thật là `O(log k)`.

Đo thử: đặt một giá trị max nằm **dưới đáy** stack, gọi xen kẽ `push`/`popMax`/`peekMax`, cuối cùng `popMax` lấy chính max dưới đáy — cách 1 trả giá ở mỗi `peekMax`, cách 2 phải "đào" cả stack ở lần cuối.

In [14]:
import time, random
rng = random.Random(716)

def bench(cls, n=4000):
    obj = cls()
    obj.push(10**7)                      # max nằm dưới đáy
    for _ in range(n):
        obj.push(rng.randint(-10**7, 10**6))
    t0 = time.perf_counter()
    for _ in range(300):
        obj.push(10**7)                  # lại một max...
        obj.popMax()                     # ...lấy ra (gần đỉnh, nhanh)
        obj.peekMax()
    obj.popMax()                         # max dưới đáy: cách 2 phải đào cả stack
    return time.perf_counter() - t0

for cls in [MaxStackList, MaxStackTwo, MaxStack, MaxStackDLL]:
    print(f"{cls.__name__:14} {bench(cls):.4f}s")

MaxStackList   0.0218s
MaxStackTwo    0.0008s
MaxStack       0.0003s
MaxStackDLL    0.0001s


## 12. Bộ test — mẫu, biên, phản ví dụ, đối chiếu

In [15]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:34} PASS -> {actual}")

check("mẫu", run_ops(MaxStack, OPS, ARGS), EXP)
check("1 phần tử", run_ops(MaxStack, ["MaxStack", "push", "peekMax", "popMax"], [[], [-3], [], []]), [None, None, -3, -3])
check("max trùng: popMax lấy gần đỉnh",
      run_ops(MaxStack, ["MaxStack", "push", "push", "push", "popMax", "top", "pop", "top"],
              [[], [7], [1], [7], [], [], [], []]),
      [None, None, None, None, 7, 1, 1, 7])
check("số âm", run_ops(MaxStack, ["MaxStack", "push", "push", "peekMax", "pop", "peekMax"],
                         [[], [-5], [-9], [], [], []]), [None, None, None, -5, -9, -5])
check("pop rồi popMax cùng phần tử ma",
      run_ops(MaxStack, ["MaxStack", "push", "push", "pop", "peekMax", "push", "popMax", "top"],
              [[], [1], [9], [], [], [4], [], []]),
      [None, None, None, 9, 1, None, 4, 1])

mẫu                                PASS -> [None, None, None, None, 5, 5, 1, 5, 1, 5]
1 phần tử                          PASS -> [None, None, -3, -3]
max trùng: popMax lấy gần đỉnh     PASS -> [None, None, None, None, 7, 1, 1, 7]
số âm                              PASS -> [None, None, None, -5, -9, -5]
pop rồi popMax cùng phần tử ma     PASS -> [None, None, None, 9, 1, None, 4, 1]


In [16]:
impls = [MaxStackList, MaxStackTwo, MaxStackHeap, MaxStack, MaxStackDLL]
for trial in range(300):
    objs = [c() for c in impls]
    ref = []                                 # oracle: list + quét
    for _ in range(60):
        ops_avail = ["push"] + (["pop", "top", "peekMax", "popMax"] if ref else [])
        op = rng.choice(ops_avail)
        if op == "push":
            x = rng.randint(-5, 5)
            ref.append(x)
            for o in objs:
                o.push(x)
            continue
        if op == "pop":
            exp = ref.pop()
        elif op == "top":
            exp = ref[-1]
        elif op == "peekMax":
            exp = max(ref)
        else:
            exp = max(ref)
            j = len(ref) - 1 - ref[::-1].index(exp)
            del ref[j]
        for o in objs:
            assert getattr(o, op)() == exp, (type(o).__name__, op)
print("300 kịch bản ngẫu nhiên (nhiều giá trị trùng): 5 cài đặt khớp oracle")

300 kịch bản ngẫu nhiên (nhiều giá trị trùng): 5 cài đặt khớp oracle


## 13. Lỗi người mới hay gặp

### Lỗi 1 — `popMax` xóa max **đầu tiên** (gần đáy) thay vì gần đỉnh

In [17]:
class WrongFirstMax(MaxStackList):
    def popMax(self):
        m = max(self.st)
        self.st.remove(m)            # list.remove xóa lần xuất hiện ĐẦU TIÊN
        return m

ops3 = ["MaxStack", "push", "push", "push", "popMax", "top"]
args3 = [[], [5], [1], [5], [], []]
print("sai :", run_ops(WrongFirstMax, ops3, args3))
print("đúng:", run_ops(MaxStack, ops3, args3))

sai : [None, None, None, None, 5, 5]
đúng: [None, None, None, None, 5, 1]


Vì sao sai: stack còn `[1, 5]` thay vì `[5, 1]` → `top()` trả `5` thay vì `1`. Sửa: tìm từ **đỉnh** xuống (chỉ số cuối).

### Lỗi 2 — Heap chỉ lưu `-x`, không có id
Khi hai phần tử cùng giá trị, ta không biết bản nào trong heap ứng với bản nào trong stack; đánh dấu "đã xóa giá trị 5" sẽ làm mất **cả hai** số 5. Sửa: mỗi lần push gán `id` duy nhất.

### Lỗi 3 — Phá hòa bằng `id` thay vì `-id`

In [18]:
class WrongTieBreak(MaxStackHeap):
    def push(self, x):
        self.stack.append((x, self.cnt))
        heapq.heappush(self.heap, (-x, self.cnt))     # SAI: id nhỏ (cũ) được ưu tiên
        self.cnt += 1
    def _clean_heap(self):
        while self.heap and self.heap[0][1] in self.removed:
            self.removed.discard(heapq.heappop(self.heap)[1])
    def popMax(self):
        self._clean_heap()
        x, i = heapq.heappop(self.heap)
        self.removed.add(i)
        return -x

print("sai :", run_ops(WrongTieBreak, ops3, args3))
print("đúng:", run_ops(MaxStack, ops3, args3))

sai : [None, None, None, None, 5, 5]
đúng: [None, None, None, None, 5, 1]


Vì sao sai: `(-5, 0) < (-5, 2)` nên heap lấy số 5 **cũ** (id 0, ở đáy). Sửa: lưu `-id` để id lớn (mới, gần đỉnh) đứng trước.

### Lỗi 4 — Quên dọn ma trước khi đọc đỉnh
Nếu `top()` trả `self.stack[-1][0]` ngay mà không `_clean_stack()`, sau `popMax` sẽ trả phần tử đã bị xóa (ở ví dụ mẫu: trả `5` thay vì `1`).

### Lỗi 5 — Hai stack: `popMax` không push lại buffer
Quên vòng `while buffer: self.push(buffer.pop())` → mất luôn các phần tử nằm trên max.

## 14. Mẹo nhận dạng pattern và biến thể

- **"Stack + truy vấn min/max `O(1)`"** → stack phụ lưu min/max tiền tố (155 Min Stack).
- **"Cần xóa phần tử bất kỳ khỏi heap"** → **xóa lười** bằng id + set đánh dấu (cũng dùng trong 480 Sliding Window Median, 2034 Stock Price Fluctuation).
- **"Nhiều phần tử bằng nhau, cần chọn cái mới nhất/cũ nhất"** → thêm timestamp `id` vào khóa heap: `(ưu_tiên, -id)` hoặc `(ưu_tiên, id)`.
- Bài liên quan: 155 Min Stack, 895 Maximum Frequency Stack, 239 Sliding Window Maximum (deque đơn điệu).

### 14.1. Biến thể A — Min Stack (155) với stack phụ

In [19]:
class MinStack:
    def __init__(self):
        self.st, self.mn = [], []
    def push(self, x):
        self.st.append(x)
        self.mn.append(x if not self.mn else min(x, self.mn[-1]))
    def pop(self):
        self.mn.pop(); self.st.pop()
    def top(self):
        return self.st[-1]
    def getMin(self):
        return self.mn[-1]

ms = MinStack()
for x in [-2, 0, -3]:
    ms.push(x)
print(ms.getMin(), end=" "); ms.pop(); print(ms.top(), ms.getMin())   # -3 0 -2

-3 0 -2


### 14.2. Biến thể B — Max của cửa sổ trượt bằng deque đơn điệu (239)
Giữ deque các **chỉ số** với giá trị giảm dần; đầu deque luôn là max của cửa sổ.

In [20]:
from collections import deque

def max_sliding_window(nums, k):
    dq, res = deque(), []
    for i, x in enumerate(nums):
        while dq and nums[dq[-1]] <= x:
            dq.pop()
        dq.append(i)
        if dq[0] <= i - k:
            dq.popleft()
        if i >= k - 1:
            res.append(nums[dq[0]])
    return res

print(max_sliding_window([1, 3, -1, -3, 5, 3, 6, 7], 3))   # [3, 3, 5, 5, 6, 7]

[3, 3, 5, 5, 6, 7]


## 15. Bài tập tự luyện

**Bài 1 (tính tay).** Lệnh: `push(2), push(8), push(3), push(8), popMax(), pop(), peekMax(), popMax(), top()`. Ghi list các giá trị **trả về** của 5 lệnh cuối vào `ex1_results`.

**Bài 2 (đoán output).** Với heap chứa `(-8, -1), (-8, -3), (-3, -2)`, `heap[0]` là gì? Gán vào `ex2_top`.

**Bài 3 (TODO).** Viết lại `Ex3MaxStack` theo cách **hai stack**.

**Bài 4 (TODO).** Viết lại `Ex4MaxStack` theo cách **heap + xóa lười**.

**Bài 5 (biến thể).** `Ex5MinMaxStack`: stack có cả `getMin()` và `getMax()` đều `O(1)` (không cần popMax).

**Bài 6 (mini challenge).** `Ex6FreqStack` (LeetCode 895): `push(x)`, `pop()` trả và xóa phần tử **xuất hiện nhiều nhất** trong stack; hòa thì lấy phần tử gần đỉnh nhất. Gợi ý: heap với khóa `(-freq, -id)`.

In [21]:
ex1_results = None   # TODO
ex2_top = None       # TODO

class Ex3MaxStack:
    def __init__(self): pass       # TODO
    def push(self, x): pass        # TODO
    def pop(self): pass            # TODO
    def top(self): pass            # TODO
    def peekMax(self): pass        # TODO
    def popMax(self): pass         # TODO

# assert run_ops(Ex3MaxStack, OPS, ARGS) == EXP

In [22]:
class Ex4MaxStack:
    def __init__(self): pass       # TODO
    def push(self, x): pass        # TODO
    def pop(self): pass            # TODO
    def top(self): pass            # TODO
    def peekMax(self): pass        # TODO
    def popMax(self): pass         # TODO

class Ex5MinMaxStack:
    def __init__(self): pass       # TODO
    def push(self, x): pass        # TODO
    def pop(self): pass            # TODO
    def getMin(self): pass         # TODO
    def getMax(self): pass         # TODO

class Ex6FreqStack:
    def __init__(self): pass       # TODO
    def push(self, x): pass        # TODO
    def pop(self): pass            # TODO

# assert run_ops(Ex4MaxStack, OPS, ARGS) == EXP
# fs = Ex6FreqStack(); [fs.push(x) for x in [5, 7, 5, 7, 4, 5]]
# assert [fs.pop() for _ in range(4)] == [5, 7, 5, 4]

## 16. Gợi ý 3 mức

- **Mức 1:** Stack chính lo thứ tự; cấu trúc phụ lo max. Khi xóa ở một nơi, nơi kia phải biết (đồng bộ ngay hoặc xóa lười).
- **Mức 2:** Bài 5: hai stack phụ `mn`, `mx` lưu min/max tiền tố. Bài 6: `freq` đếm số lần mỗi giá trị đang có; mỗi `push(x)` tăng `freq[x]` và đẩy `(-freq[x], -id, x)` vào heap; `pop` lấy đỉnh heap, giảm `freq[x]`. Không cần xóa lười vì mỗi bản ghi heap ứng với đúng một "tầng tần suất".
- **Mức 3:** Bài 1 — vẽ stack sau từng lệnh: `[2,8,3,8]` → popMax lấy 8 gần đỉnh → `[2,8,3]` → pop 3 → `[2,8]` → ...

## 17. Đáp án tham khảo

Bài 1: `[2,8,3,8]` → `popMax()=8` → `[2,8,3]` → `pop()=3` → `[2,8]` → `peekMax()=8` → `popMax()=8` → `[2]` → `top()=2`. Vậy `ex1_results = [8, 3, 8, 8, 2]`.
Bài 2: `(-8, -3)` (cùng `-8`, so tiếp `-3 < -1`).

In [23]:
ans_ex1_results = [8, 3, 8, 8, 2]
ans_ex2_top = (-8, -3)

class AnsEx3MaxStack(MaxStackTwo):
    pass                      # đúng bằng cách 2 ở trên (tự viết lại để luyện)

class AnsEx4MaxStack(MaxStackHeap):
    pass                      # đúng bằng cách 3 ở trên

class AnsEx5MinMaxStack:
    def __init__(self):
        self.st, self.mn, self.mx = [], [], []
    def push(self, x):
        self.st.append(x)
        self.mn.append(x if not self.mn else min(x, self.mn[-1]))
        self.mx.append(x if not self.mx else max(x, self.mx[-1]))
    def pop(self):
        self.mn.pop(); self.mx.pop()
        return self.st.pop()
    def getMin(self):
        return self.mn[-1]
    def getMax(self):
        return self.mx[-1]

class AnsEx6FreqStack:
    def __init__(self):
        self.freq, self.heap, self.cnt = {}, [], 0
    def push(self, x):
        self.freq[x] = self.freq.get(x, 0) + 1
        heapq.heappush(self.heap, (-self.freq[x], -self.cnt, x))
        self.cnt += 1
    def pop(self):
        _, _, x = heapq.heappop(self.heap)
        self.freq[x] -= 1
        return x

o = MaxStack()
for x in [2, 8, 3, 8]:
    o.push(x)
assert [o.popMax(), o.pop(), o.peekMax(), o.popMax(), o.top()] == ans_ex1_results
h = [(-8, -1), (-8, -3), (-3, -2)]; heapq.heapify(h)
assert h[0] == ans_ex2_top
assert run_ops(AnsEx3MaxStack, OPS, ARGS) == EXP
assert run_ops(AnsEx4MaxStack, OPS, ARGS) == EXP
mm = AnsEx5MinMaxStack()
for x in [3, -1, 7, 2]:
    mm.push(x)
assert (mm.getMin(), mm.getMax()) == (-1, 7)
mm.pop(); mm.pop()
assert (mm.getMin(), mm.getMax()) == (-1, 3)
fs = AnsEx6FreqStack()
for x in [5, 7, 5, 7, 4, 5]:
    fs.push(x)
assert [fs.pop() for _ in range(4)] == [5, 7, 5, 4]
print("Đáp án tham khảo: PASS toàn bộ")

Đáp án tham khảo: PASS toàn bộ


## 18. Template nhớ nhanh và checklist

```python
import heapq

class MaxStack:
    def __init__(self):
        self.stack, self.heap, self.removed, self.cnt = [], [], set(), 0
    def push(self, x):
        self.stack.append((x, self.cnt))
        heapq.heappush(self.heap, (-x, -self.cnt))
        self.cnt += 1
    def _clean_stack(self):
        while self.stack and self.stack[-1][1] in self.removed:
            self.removed.discard(self.stack.pop()[1])
    def _clean_heap(self):
        while self.heap and -self.heap[0][1] in self.removed:
            self.removed.discard(-heapq.heappop(self.heap)[1])
    def pop(self):
        self._clean_stack(); x, i = self.stack.pop(); self.removed.add(i); return x
    def top(self):
        self._clean_stack(); return self.stack[-1][0]
    def peekMax(self):
        self._clean_heap(); return -self.heap[0][0]
    def popMax(self):
        self._clean_heap(); x, i = heapq.heappop(self.heap); self.removed.add(-i); return -x
```

Câu thần chú: **"Hai cấu trúc + id duy nhất + xóa lười; dọn ma trước khi đọc đỉnh; hòa thì id lớn thắng."**

### Checklist trước khi submit
- [ ] `popMax` với max trùng lấy phần tử **gần đỉnh nhất**.
- [ ] Heap lưu `(-x, -id)` (max trước, id mới trước).
- [ ] `top`, `pop`, `peekMax`, `popMax` đều dọn ma trước khi đọc.
- [ ] Sau khi dọn ma thì `discard` id khỏi `removed`.
- [ ] Test: max trùng, số âm, `pop` rồi `peekMax`, một phần tử.
- [ ] Nói được: heap + lazy `O(log n)` amortized; hai stack thì `popMax` `O(n)`.